# 📊 Chronos Data Validation & Zero-Shot Baseline
    Notebook ini digunakan untuk:
    1. Memuat data time-series
    2. Memformatnya menjadi objek GluonTS (format standar Chronos)
    3. Menjalankan inferensi 'Zero-Shot' untuk melihat baseline performa

---
### `Import Library`

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from chronos import ChronosPipeline
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

### `Load Dataset`

In [2]:
df = pd.read_csv('../data/preprocessed_no_resample.csv')
df.head()

,Timestamp,Impact,Priority,incident
0,2024-01-01 08:13:45,2,2,1
1,2024-01-10 15:06:49,2,2,1
2,2024-01-05 06:11:00,1,1,0
3,2024-01-10 14:19:44,1,1,0
4,2024-01-10 14:02:29,1,1,0


### `Data Preparation`

In [4]:
# Set Timestamp as index
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.set_index('Timestamp')

# Ensure timestamp alignment between features and labels
ts_resampled = df['Priority'].resample('h').count().fillna(0)
label_resampled = df['incident'].resample('h').max().fillna(0)

# Align both series to same index
aligned_data = pd.DataFrame({
    'priority': ts_resampled,
    'incident': label_resampled
}).dropna()  # Remove any misaligned rows

full_values = aligned_data['priority'].values
full_labels = aligned_data['incident'].values.astype(int)
total_len = len(full_values)

print(f"Total aligned data points: {total_len}")
print(f"Incident rate: {full_labels.mean():.2%}\n")

# --- SPLIT: Reserve last 20% for validation ---
train_size = int(total_len * 0.8)
print(f"Using data up to index {train_size} for context (80%)")
print(f"Validation windows from index {train_size} onward (20%)\n")

Total aligned data points: 7321
Incident rate: 45.46%

Using data up to index 5856 for context (80%)
Validation windows from index 5856 onward (20%)



### `Model Configuration`

In [10]:
# Load model
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-tiny",
    device_map="cuda",
    dtype=torch.bfloat16,
)

In [5]:
# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

In [6]:
# Backtesting parameters
num_windows = 5          # Jumlah validation window
prediction_length = 48   # Panjang prediksi per window (48 jam)
context_length = 300     # Panjang konteks history yang dilihat model
threshold = 0.60         # Threshold score untuk menentukan Insiden

# Container untuk menampung semua hasil dari 5 window
all_y_true = []
all_y_scores = []
all_y_pred = []

In [11]:
print(f"🚀 Memulai Backtesting pada {num_windows} validation window (@ {prediction_length} jam)...")
print(f"Threshold yang digunakan: > {threshold}\n")

# --- BACKTESTING LOOP ---
for i in range(num_windows):
    # Calculate non-overlapping windows moving backwards from end
    end_idx = total_len - (i * prediction_length)
    start_idx = end_idx - prediction_length
    context_start_idx = start_idx - context_length
    
    # Ensure validation window is in test set
    if start_idx < train_size:
        print(f"⚠️ Window {i+1}: Enters training data. Stopping.")
        break
        
    if context_start_idx < 0:
        print(f"⚠️ Window {i+1}: Insufficient historical data. Stopping.")
        break
    
    # 1. Prepare context and labels
    context_window = torch.tensor(full_values[context_start_idx:start_idx], dtype=torch.float32)
    y_true_window = full_labels[start_idx:end_idx]
    
    # 2. Run prediction
    with torch.no_grad():
        forecast = pipeline.predict(
            context_window,
            prediction_length=prediction_length,
            num_samples=20
        )
    
    # 3. Extract median
    median = torch.quantile(forecast, torch.tensor(0.5), dim=1).squeeze().cpu().numpy()
    
    # 4. Convert to binary predictions
    y_pred_window = (median > threshold).astype(int)
    
    # 5. Store results
    all_y_true.extend(y_true_window)
    all_y_scores.extend(median)
    all_y_pred.extend(y_pred_window)
    
    print(f"Window {i+1} [{context_start_idx}→{start_idx}→{end_idx}]: "
          f"{int(np.sum(y_true_window))} incidents")

🚀 Memulai Backtesting pada 5 validation window (@ 48 jam)...
Threshold yang digunakan: > 0.6

Window 1 [6973→7273→7321]: 23 incidents
Window 2 [6925→7225→7273]: 23 incidents
Window 3 [6877→7177→7225]: 18 incidents
Window 4 [6829→7129→7177]: 24 incidents
Window 5 [6781→7081→7129]: 25 incidents


### `Evaluation`

In [14]:
print("\n" + "="*40)
print("HASIL AKHIR VALIDASI (RATA-RATA 5 WINDOW)")
print("="*40)

# Pastikan array numpy
all_y_true = np.array(all_y_true)
all_y_scores = np.array(all_y_scores)
all_y_pred = np.array(all_y_pred)

# Hitung Metrik
acc = accuracy_score(all_y_true, all_y_pred)
prec = precision_score(all_y_true, all_y_pred, zero_division=0)
rec = recall_score(all_y_true, all_y_pred, zero_division=0)
f1 = f1_score(all_y_true, all_y_pred, zero_division=0)

print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall:     {rec:.4f}")
print(f"F1-Score:   {f1:.4f}")

try:
    roc_auc = roc_auc_score(all_y_true, all_y_scores)
    pr_auc = average_precision_score(all_y_true, all_y_scores)
    print(f"ROC-AUC:    {roc_auc:.4f}")
    print(f"PR-AUC:     {pr_auc:.4f}")
except:
    print("ROC/PR AUC: Tidak dapat dihitung (Data label mungkin homogen)")

print("-" * 20)
cm = confusion_matrix(all_y_true, all_y_pred)
print(f"Total Confusion Matrix ({len(all_y_true)} data points):")
print(f"TN: {cm[0,0]} | FP: {cm[0,1]}")
print(f"FN: {cm[1,0]} | TP: {cm[1,1]}")


HASIL AKHIR VALIDASI (RATA-RATA 5 WINDOW)
Accuracy:   0.7292
Precision:  0.6558
Recall:     0.8938
F1-Score:   0.7566
ROC-AUC:    0.7938
PR-AUC:     0.7472
--------------------
Total Confusion Matrix (240 data points):
TN: 74 | FP: 53
FN: 12 | TP: 101
